# PySpark 4.0.2 + Delta Lake 4.0.0 en Google Colab con datos reales (NYC TLC)

Este notebook está diseñado para ejecutarse en **Google Colab** usando:

- `pyspark==4.0.2`
- `delta-spark==4.0.0`

Usa datos públicos reales del portal **NYC TLC Trip Record Data** en formato Parquet.

## Objetivos
1. Instalar y configurar Spark + Delta en Colab.
2. Descargar dos meses de datos reales.
3. Leer y transformar datos con PySpark.
4. Escribir y consultar tablas Delta Lake.
5. Ejecutar una actualización tipo upsert con `MERGE`.
6. Comparar de forma básica Parquet vs Delta.

> Sugerencia: en Colab, ejecuta las celdas en orden. Si ves errores de entorno, reinicia el runtime y vuelve a ejecutar.

In [1]:
!pip install -q pyspark==4.0.2 delta-spark==4.0.0

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

try:
    spark.stop()
except:
    pass

builder = (
    SparkSession.builder
    .appName("EAFIT-PySpark-4.0.2-Delta-4.0.0-Colab")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "4g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.2


## Descargar datos reales de NYC TLC

El portal oficial NYC TLC publica mensualmente los datos en **Parquet**. Aquí descargamos dos meses para tener un volumen razonable para Colab.

In [3]:
import os
import urllib.request

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data"
files = [
    "yellow_tripdata_2024-01.parquet",
    "yellow_tripdata_2024-02.parquet",
]

download_dir = "/content/nyc_tlc_raw"
os.makedirs(download_dir, exist_ok=True)

local_files = []
for f in files:
    url = f"{base_url}/{f}"
    local_path = os.path.join(download_dir, f)
    if not os.path.exists(local_path):
        print("Descargando:", url)
        urllib.request.urlretrieve(url, local_path)
    else:
        print("Ya existe:", local_path)
    local_files.append(local_path)

print("\nArchivos locales:")
for p in local_files:
    print(" -", p, "|", round(os.path.getsize(p)/1024/1024, 2), "MB")

Descargando: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
Descargando: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet

Archivos locales:
 - /content/nyc_tlc_raw/yellow_tripdata_2024-01.parquet | 47.65 MB
 - /content/nyc_tlc_raw/yellow_tripdata_2024-02.parquet | 48.02 MB


## Leer los archivos Parquet con Spark

In [4]:
from pyspark.sql import functions as F

base_df = (
    spark.read.parquet(*local_files)
    .withColumn("year", F.year("tpep_pickup_datetime"))
    .withColumn("month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
)

print("Número de columnas:", len(base_df.columns))
base_df.printSchema()

Número de columnas: 22
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- pickup_date: date (null

In [5]:
print("Número de registros:", base_df.count())
base_df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type",
    "year",
    "month"
).show(10, truncate=False)

Número de registros: 5972150
+--------+--------------------+---------------------+---------------+-------------+-----------+------------+------------+----+-----+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|total_amount|payment_type|year|month|
+--------+--------------------+---------------------+---------------+-------------+-----------+------------+------------+----+-----+
|2       |2024-02-01 00:04:45 |2024-02-01 00:19:58  |1              |4.39         |20.5       |26.78       |1           |2024|2    |
|2       |2024-02-01 00:56:31 |2024-02-01 01:10:53  |1              |7.71         |31.0       |45.0        |1           |2024|2    |
|2       |2024-02-01 00:07:50 |2024-02-01 00:43:12  |2              |28.69        |70.0       |82.69       |2           |2024|2    |
|1       |2024-02-01 00:01:49 |2024-02-01 00:10:47  |1              |1.1          |9.3        |17.15       |1           |2024|2    |
|1       |2024-02-01 00:37:35 |2024-02-0

## Limpieza mínima y selección de columnas

Para el laboratorio, filtramos valores claramente inválidos y dejamos un subconjunto manejable de columnas.

In [6]:
trips = (
    base_df
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("total_amount") > 0)
    .select(
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type",
        "PULocationID",
        "DOLocationID",
        "year",
        "month",
        "pickup_date"
    )
)

trips.cache()
print("Registros limpios:", trips.count())
trips.show(5, truncate=False)

Registros limpios: 5771187
+--------+--------------------+---------------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+----+-----+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|PULocationID|DOLocationID|year|month|pickup_date|
+--------+--------------------+---------------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+----+-----+-----------+
|2       |2024-02-01 00:04:45 |2024-02-01 00:19:58  |1              |4.39         |20.5       |1.28      |26.78       |1           |68          |236         |2024|2    |2024-02-01 |
|2       |2024-02-01 00:56:31 |2024-02-01 01:10:53  |1              |7.71         |31.0       |9.0       |45.0        |1           |48          |243         |2024|2    |2024-02-01 |
|2       |2024-02-01 00:07:50 |2024-02-01 00:43:12  |2         

## Exploración básica

In [7]:
daily_stats = (
    trips.groupBy("pickup_date")
    .agg(
        F.count("*").alias("num_trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount")
    )
    .orderBy(F.desc("num_trips"))
)

daily_stats.show(10, truncate=False)

+-----------+---------+-----------------+----------------+
|pickup_date|num_trips|avg_trip_distance|avg_total_amount|
+-----------+---------+-----------------+----------------+
|2024-02-29 |124177   |3.44             |27.87           |
|2024-02-15 |117700   |3.07             |27.1            |
|2024-02-14 |117694   |4.84             |26.7            |
|2024-02-24 |113025   |3.0              |25.28           |
|2024-02-27 |112524   |3.23             |27.01           |
|2024-02-22 |112035   |5.24             |27.82           |
|2024-02-08 |110718   |4.22             |28.03           |
|2024-02-28 |108191   |3.56             |27.89           |
|2024-02-10 |108062   |5.08             |25.37           |
|2024-01-27 |107219   |2.88             |25.08           |
+-----------+---------+-----------------+----------------+
only showing top 10 rows


In [8]:
payment_stats = (
    trips.groupBy("payment_type")
    .agg(
        F.count("*").alias("num_trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("total_amount"), 2).alias("avg_total")
    )
    .orderBy(F.desc("num_trips"))
)

payment_stats.show(truncate=False)

+------------+---------+--------+-------+---------+
|payment_type|num_trips|avg_fare|avg_tip|avg_total|
+------------+---------+--------+-------+---------+
|1           |4620633  |18.35   |4.15   |28.04    |
|2           |819380   |18.55   |0.0    |23.79    |
|0           |264099   |19.94   |1.57   |25.79    |
|4           |46071    |19.83   |0.01   |25.22    |
|3           |21004    |17.45   |0.01   |22.56    |
+------------+---------+--------+-------+---------+



## Escribir como tabla Delta Lake

Particionamos por `year` y `month`.

In [9]:
delta_path = "/content/delta/nyc_taxi_yellow"

import shutil, os
if os.path.exists(delta_path):
    shutil.rmtree(delta_path)

(
    trips.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")
    .save(delta_path)
)

print("Tabla Delta creada en:", delta_path)

Tabla Delta creada en: /content/delta/nyc_taxi_yellow


In [10]:
delta_df = spark.read.format("delta").load(delta_path)

print("Registros en Delta:", delta_df.count())
delta_df.show(5, truncate=False)

Registros en Delta: 5771187
+--------+--------------------+---------------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+----+-----+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|payment_type|PULocationID|DOLocationID|year|month|pickup_date|
+--------+--------------------+---------------------+---------------+-------------+-----------+----------+------------+------------+------------+------------+----+-----+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |17.7       |0.0       |22.7        |2           |186         |79          |2024|1    |2024-01-01 |
|1       |2024-01-01 00:03:00 |2024-01-01 00:09:36  |1              |1.8          |10.0       |3.75      |18.75       |1           |140         |236         |2024|1    |2024-01-01 |
|1       |2024-01-01 00:17:06 |2024-01-01 00:35:01  |1        

## Consultas SQL sobre Delta

In [11]:
delta_df.createOrReplaceTempView("yellow_trips_delta")

query = '''
SELECT
  year,
  month,
  COUNT(*) AS num_trips,
  ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
  ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM yellow_trips_delta
GROUP BY year, month
ORDER BY year, month
'''

spark.sql(query).show(truncate=False)

+----+-----+---------+-----------------+----------------+
|year|month|num_trips|avg_trip_distance|avg_total_amount|
+----+-----+---------+-----------------+----------------+
|2002|12   |1        |0.63             |10.5            |
|2008|12   |1        |1.62             |19.9            |
|2009|1    |4        |5.73             |36.21           |
|2023|12   |10       |2.6              |22.46           |
|2024|1    |2869707  |3.73             |27.35           |
|2024|2    |2901462  |3.96             |27.24           |
|2024|3    |2        |5.55             |43.67           |
+----+-----+---------+-----------------+----------------+



## Ver el historial Delta

In [12]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)
delta_table.history().show(truncate=False)

+-------+-----------------------+------+--------+---------+----------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                                 |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                   |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+----------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------------------------------------------------------------+------------+--------

## MERGE (upsert) de ejemplo

Creamos un pequeño DataFrame con correcciones simuladas y lo fusionamos contra la tabla Delta.

In [13]:
from pyspark.sql import Row

sample_keys = (
    delta_df
    .select("VendorID", "tpep_pickup_datetime", "PULocationID", "DOLocationID")
    .limit(5)
    .collect()
)

updates_data = []
for r in sample_keys:
    updates_data.append(Row(
        VendorID=r["VendorID"],
        tpep_pickup_datetime=r["tpep_pickup_datetime"],
        PULocationID=r["PULocationID"],
        DOLocationID=r["DOLocationID"],
        passenger_count=9,
        tip_amount=99.99
    ))

updates_df = spark.createDataFrame(updates_data)
updates_df.show(truncate=False)

+--------+--------------------+------------+------------+---------------+----------+
|VendorID|tpep_pickup_datetime|PULocationID|DOLocationID|passenger_count|tip_amount|
+--------+--------------------+------------+------------+---------------+----------+
|2       |2024-01-01 00:57:55 |186         |79          |9              |99.99     |
|1       |2024-01-01 00:03:00 |140         |236         |9              |99.99     |
|1       |2024-01-01 00:17:06 |236         |79          |9              |99.99     |
|1       |2024-01-01 00:36:38 |79          |211         |9              |99.99     |
|1       |2024-01-01 00:46:51 |211         |148         |9              |99.99     |
+--------+--------------------+------------+------------+---------------+----------+



In [14]:
target = DeltaTable.forPath(spark, delta_path)

merge_condition = '''
t.VendorID = s.VendorID AND
t.tpep_pickup_datetime = s.tpep_pickup_datetime AND
t.PULocationID = s.PULocationID AND
t.DOLocationID = s.DOLocationID
'''

(
    target.alias("t")
    .merge(updates_df.alias("s"), merge_condition)
    .whenMatchedUpdate(set={
        "passenger_count": "s.passenger_count",
        "tip_amount": "s.tip_amount"
    })
    .execute()
)

print("MERGE ejecutado correctamente.")

MERGE ejecutado correctamente.


In [15]:
spark.read.format("delta").load(delta_path).select(
    "VendorID", "tpep_pickup_datetime", "PULocationID", "DOLocationID",
    "passenger_count", "tip_amount"
).show(10, truncate=False)

+--------+--------------------+------------+------------+---------------+----------+
|VendorID|tpep_pickup_datetime|PULocationID|DOLocationID|passenger_count|tip_amount|
+--------+--------------------+------------+------------+---------------+----------+
|2       |2024-02-01 00:04:45 |68          |236         |1              |1.28      |
|2       |2024-02-01 00:56:31 |48          |243         |1              |9.0       |
|2       |2024-02-01 00:07:50 |132         |261         |2              |0.0       |
|1       |2024-02-01 00:01:49 |161         |163         |1              |2.85      |
|1       |2024-02-01 00:37:35 |246         |79          |1              |0.0       |
|1       |2024-02-01 00:55:17 |79          |4           |1              |2.55      |
|2       |2024-02-01 00:04:53 |249         |163         |1              |3.84      |
|2       |2024-02-01 00:35:00 |163         |151         |1              |3.56      |
|2       |2024-02-01 00:00:15 |246         |48          |1       

## Comparación básica Parquet vs Delta

No es un benchmark riguroso, pero sí una comparación didáctica útil en Colab.

In [16]:
import time

t0 = time.time()
parquet_count = spark.read.parquet(*local_files).count()
t1 = time.time()

t2 = time.time()
delta_count = spark.read.format("delta").load(delta_path).count()
t3 = time.time()

print(f"Parquet count: {parquet_count:,} | tiempo: {t1 - t0:.2f} s")
print(f"Delta   count: {delta_count:,} | tiempo: {t3 - t2:.2f} s")

Parquet count: 5,972,150 | tiempo: 0.50 s
Delta   count: 5,771,187 | tiempo: 1.86 s


In [17]:
parquet_df = spark.read.parquet(*local_files).withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

t0 = time.time()
parquet_result = (
    parquet_df
    .groupBy("pickup_date")
    .agg(F.count("*").alias("num_trips"), F.avg("total_amount").alias("avg_total"))
    .orderBy(F.desc("num_trips"))
)
parquet_result.show(5, truncate=False)
t1 = time.time()

t2 = time.time()
delta_result = (
    spark.read.format("delta").load(delta_path)
    .groupBy("pickup_date")
    .agg(F.count("*").alias("num_trips"), F.avg("total_amount").alias("avg_total"))
    .orderBy(F.desc("num_trips"))
)
delta_result.show(5, truncate=False)
t3 = time.time()

print(f"Consulta agregada Parquet: {t1 - t0:.2f} s")
print(f"Consulta agregada Delta  : {t3 - t2:.2f} s")

+-----------+---------+------------------+
|pickup_date|num_trips|avg_total         |
+-----------+---------+------------------+
|2024-02-29 |129890   |27.23810709061438 |
|2024-02-15 |122956   |26.459873531994713|
|2024-02-14 |121771   |26.128292614825718|
|2024-02-27 |118614   |26.288264370140983|
|2024-02-24 |117831   |24.684210352114395|
+-----------+---------+------------------+
only showing top 5 rows
+-----------+---------+------------------+
|pickup_date|num_trips|avg_total         |
+-----------+---------+------------------+
|2024-02-29 |124177   |27.871682356635176|
|2024-02-15 |117700   |27.10477748513129 |
|2024-02-14 |117694   |26.699766768058925|
|2024-02-24 |113025   |25.275474452554548|
|2024-02-27 |112524   |27.01349143293775 |
+-----------+---------+------------------+
only showing top 5 rows
Consulta agregada Parquet: 2.46 s
Consulta agregada Delta  : 3.50 s


## Archivos Delta generados

La carpeta Delta incluye:

- archivos de datos Parquet
- la carpeta `_delta_log` con el historial transaccional

Eso es lo que diferencia a Delta Lake de un conjunto simple de archivos Parquet.

In [ ]:
import os

for root, dirs, files in os.walk(delta_path):
    level = root.replace(delta_path, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for f in files[:10]:
        print(f"{subindent}{f}")

## Ejercicios sugeridos

1. Cambia los meses descargados por otros dos meses del mismo año.
2. Agrega una columna `trip_duration_minutes`.
3. Calcula el top 20 de rutas `PULocationID -> DOLocationID`.
4. Escribe una segunda tabla Delta particionada por `pickup_date`.
5. Compara filtros por `month` contra la tabla Delta particionada.

## Referencias
- Portal NYC TLC Trip Record Data.
- Documentación oficial de Apache Spark 4.0.2.
- Documentación oficial de Delta Lake 4.0.0.